In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../../data/raw/RecipeNLG_dataset.csv")
# df = pd.read_csv("")

In [3]:
df.head(1)

,Unnamed: 0,title,ingredients,directions,link,source,NER
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."


In [4]:
print(df.loc[0, "ingredients"])
print(type(df.loc[0, "ingredients"]))

print(df.loc[0, "NER"])

["1 c. firmly packed brown sugar", "1/2 c. evaporated milk", "1/2 tsp. vanilla", "1/2 c. broken nuts (pecans)", "2 Tbsp. butter or margarine", "3 1/2 c. bite size shredded rice biscuits"]
<class 'str'>
["brown sugar", "milk", "vanilla", "nuts", "butter", "bite size shredded rice biscuits"]


In [5]:
# df = df.sample(1000000, random_state=42)

In [6]:
import ast

def safe_parse(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df["ingredients_list"] = df["ingredients"].apply(safe_parse)
df["steps_list"] = df["directions"].apply(safe_parse)
df["ner_list"] = df["NER"].apply(safe_parse)

In [7]:
df[["ingredients_list", "steps_list", "ner_list"]].head(5)

,ingredients_list,steps_list,ner_list
0,"[1 c. firmly packed brown sugar, 1/2 c. evapor...","[In a heavy 2-quart saucepan, mix brown sugar,...","[brown sugar, milk, vanilla, nuts, butter, bit..."
1,"[1 small jar chipped beef, cut up, 4 boned chi...","[Place chipped beef on bottom of baking dish.,...","[beef, chicken breasts, cream of mushroom soup..."
2,"[2 (16 oz.) pkg. frozen corn, 1 (8 oz.) pkg. c...","[In a slow cooker, combine all ingredients. Co...","[frozen corn, cream cheese, butter, garlic pow..."
3,"[1 large whole chicken, 2 (10 1/2 oz.) cans ch...","[Boil and debone chicken., Put bite size piece...","[chicken, chicken gravy, cream of mushroom sou..."
4,"[1 c. peanut butter, 3/4 c. graham cracker cru...",[Combine first four ingredients and press in 1...,"[peanut butter, graham cracker crumbs, butter,..."


In [8]:
def clean_text(text):
    return str(text).strip().lower()


In [9]:
df["title"] = df["title"].apply(clean_text)

df["ingredients_clean"] = df["ner_list"].apply(
    lambda lst: [clean_text(i) for i in lst if i]
)

df["steps_clean"] = df["steps_list"].apply(
    lambda lst: [clean_text(i) for i in lst if i]
)

In [10]:
df_clean = df[
    (df["ingredients_clean"].map(len) > 2) &
    (df["steps_clean"].map(len) > 1)
]

In [11]:
import uuid

def build_record(row):
    return {
        "id": str(uuid.uuid4()),
        "title": row["title"],
        "ingredients": row["ingredients_clean"],
        "steps": row["steps_clean"]
    }

final_data = df_clean.apply(build_record, axis=1).tolist()

In [12]:
final_data[0]

{'id': '89feba95-f1a0-497a-873f-2105c4ef927e',
 'title': 'no-bake nut cookies',
 'ingredients': ['brown sugar',
  'milk',
  'vanilla',
  'nuts',
  'butter',
  'bite size shredded rice biscuits'],
 'steps': ['in a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine.',
  'stir over medium heat until mixture bubbles all over top.',
  'boil and stir 5 minutes more. take off heat.',
  'stir in vanilla and cereal; mix well.',
  'using 2 teaspoons, drop and shape into 30 clusters on wax paper.',
  'let stand until firm, about 30 minutes.']}

In [13]:
import json

with open("../../data/processed/recipes_cleaned.json", "w") as f:
    json.dump(final_data, f, indent=2)

print(f"Saved {len(final_data)} recipes")

Saved 2021247 recipes
